In [1]:
import sys
sys.path.append("/media/mrsmile/IA/tesis/")
import torch
from models.resunet_segmentation import ResUNetSegmentation

CHECKPOINT = "/media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_best.pth"

model = ResUNetSegmentation()
model.load_pretrained_encoder(CHECKPOINT)

# Verifica shapes
x   = torch.randn(1, 1, 128, 128, 128)
out = model(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}")   # debe ser (1, 1, 128, 128, 128)

# Verifica grupos de LR
groups = model.get_param_groups()
for g in groups:
    n = sum(p.numel() for p in g['params'])
    print(f"{g['name']}: {n:,} parámetros — LR {g['lr']}")

   Encoder cargado: 24 tensores
   Bloques: ['e1', 'e2', 'e3', 'e4']
   Decoder: inicialización aleatoria (kaiming)

Input:  torch.Size([1, 1, 128, 128, 128])
Output: torch.Size([1, 1, 128, 128, 128])
encoder: 3,556,640 parámetros — LR 1e-05
decoder: 2,129,825 parámetros — LR 0.0001


In [2]:
import sys
sys.path.append("/media/mrsmile/IA/tesis/")
import json
from src.dataset_segmentation import CTASegmentationDataset

JSON_PATH      = "/media/mrsmile/IA/tesis/data/metadata/data_split.json"
VOL_DIR        = "/media/mrsmile/IA/tesis/data/processed/memmap/volumes"
MASK_DIR       = "/media/mrsmile/IA/tesis/data/processed/memmap/masks"
VOL_META_DIR   = "/media/mrsmile/IA/tesis/data/processed/memmap/meta"
MASK_META_DIR  = "/media/mrsmile/IA/tesis/data/processed/memmap/masks_meta"

with open(JSON_PATH) as f:
    split = json.load(f)

fine_tuning_list = split['asoca']['fine_tuning']

ds = CTASegmentationDataset(
    vol_dir       = VOL_DIR,
    mask_dir      = MASK_DIR,
    vol_meta_dir  = VOL_META_DIR,
    mask_meta_dir = MASK_META_DIR,
    file_list     = fine_tuning_list,
    patch_size    = (128, 128, 128),
    patches_per_volume = 4,
    positive_ratio = 0.5,
)

# Prueba un batch
vol_patch, mask_patch = ds[0]
print(f"Volumen patch:  {vol_patch.shape}  dtype: {vol_patch.dtype}")
print(f"Máscara patch:  {mask_patch.shape}  dtype: {mask_patch.dtype}")
print(f"Valores máscara únicos: {mask_patch.unique()}")
print(f"% positivos en parche:  {100*mask_patch.mean():.3f}%")
print(f"Total muestras dataset: {len(ds)}")

✅ Dataset segmentación: 30 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
Volumen patch:  torch.Size([1, 128, 128, 128])  dtype: torch.float32
Máscara patch:  torch.Size([1, 128, 128, 128])  dtype: torch.float32
Valores máscara únicos: tensor([0., 1.])
% positivos en parche:  0.438%
Total muestras dataset: 120


In [3]:
from src.losses import DiceBCELoss, dice_score
import torch

criterion = DiceBCELoss(dice_weight=1.0, bce_weight=1.0)

# Simula predicción perfecta
pred   = torch.ones(2, 1, 128, 128, 128) * 10.0   # logits muy altos → sigmoid ≈ 1
target = torch.ones(2, 1, 128, 128, 128)

loss, ld, lb = criterion(pred, target)
ds = dice_score(pred, target)
print(f"Predicción perfecta — Loss: {loss:.4f} | Dice: {ld:.4f} | BCE: {lb:.4f} | Score: {ds:.4f}")

# Simula predicción todo fondo
pred_zero = torch.ones(2, 1, 128, 128, 128) * -10.0   # logits muy negativos → sigmoid ≈ 0
loss2, ld2, lb2 = criterion(pred_zero, target)
ds2 = dice_score(pred_zero, target)
print(f"Todo fondo          — Loss: {loss2:.4f} | Dice: {ld2:.4f} | BCE: {lb2:.4f} | Score: {ds2:.4f}")

Predicción perfecta — Loss: 0.0001 | Dice: 0.0000 | BCE: 0.0000 | Score: 1.0000
Todo fondo          — Loss: 11.0000 | Dice: 0.9999 | BCE: 10.0000 | Score: 0.0000


In [4]:
import sys, os, json
sys.path.append("/media/mrsmile/IA/tesis/")

import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from tqdm import tqdm

from models.resunet_segmentation import ResUNetSegmentation
from src.dataset_segmentation import CTASegmentationDataset
from src.losses import DiceBCELoss, dice_score

In [5]:
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 300
BATCH_SIZE   = 2
LR_ENCODER   = 1e-5
LR_DECODER   = 1e-4
LR_MIN       = 1e-6
PATCH_SIZE   = (128, 128, 128)
PPV          = 6       # patches por volumen
POS_RATIO    = 0.5
VAL_SPLIT    = 0.2     # 20% de los 30 casos → 6 val, 24 train

In [6]:
CHECKPOINT_ENCODER = "/media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_best.pth"
JSON_PATH          = "/media/mrsmile/IA/tesis/data/metadata/data_split.json"
VOL_DIR            = "/media/mrsmile/IA/tesis/data/processed/memmap/volumes"
MASK_DIR           = "/media/mrsmile/IA/tesis/data/processed/memmap/masks"
VOL_META_DIR       = "/media/mrsmile/IA/tesis/data/processed/memmap/meta"
MASK_META_DIR      = "/media/mrsmile/IA/tesis/data/processed/memmap/masks_meta"

In [7]:
RUN_NAME    = f"seg_{datetime.now().strftime('%Y%m%d_%H%M')}"
CKPT_DIR    = f"/media/mrsmile/IA/tesis/runs/checkpoints/segmentation/{RUN_NAME}"
TB_DIR      = f"/media/mrsmile/IA/tesis/runs/tensorboard/segmentation/{RUN_NAME}"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TB_DIR,   exist_ok=True)

In [8]:
with open(JSON_PATH) as f:
    split_data = json.load(f)

fine_tuning_list = split_data['asoca']['fine_tuning']

# Dataset completo de fine-tuning
full_ds = CTASegmentationDataset(
    vol_dir            = VOL_DIR,
    mask_dir           = MASK_DIR,
    vol_meta_dir       = VOL_META_DIR,
    mask_meta_dir      = MASK_META_DIR,
    file_list          = fine_tuning_list,
    patch_size         = PATCH_SIZE,
    patches_per_volume = PPV,
    positive_ratio     = POS_RATIO,
)

# Split train/val a nivel de volumen (no de parche)
n_files   = len(full_ds.files)
n_val     = max(1, int(n_files * VAL_SPLIT))
n_train   = n_files - n_val

# Separar archivos manualmente para no mezclar volúmenes entre train y val
np.random.seed(42)
indices   = np.random.permutation(n_files)
train_files = [full_ds.files[i] for i in indices[:n_train]]
val_files   = [full_ds.files[i] for i in indices[n_train:]]

print(f"Train: {len(train_files)} volúmenes | Val: {len(val_files)} volúmenes")

train_ds = CTASegmentationDataset(
    vol_dir            = VOL_DIR,
    mask_dir           = MASK_DIR,
    vol_meta_dir       = VOL_META_DIR,
    mask_meta_dir      = MASK_META_DIR,
    file_list          = [f + ".nii.gz" for f in train_files],
    patch_size         = PATCH_SIZE,
    patches_per_volume = PPV,
    positive_ratio     = POS_RATIO,
)

val_ds = CTASegmentationDataset(
    vol_dir            = VOL_DIR,
    mask_dir           = MASK_DIR,
    vol_meta_dir       = VOL_META_DIR,
    mask_meta_dir      = MASK_META_DIR,
    file_list          = [f + ".nii.gz" for f in val_files],
    patch_size         = PATCH_SIZE,
    patches_per_volume = PPV,
    positive_ratio     = 0.5,
)

def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True,
                          worker_init_fn=seed_worker)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

✅ Dataset segmentación: 30 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
Train: 24 volúmenes | Val: 6 volúmenes
✅ Dataset segmentación: 24 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
✅ Dataset segmentación: 6 volúmenes
Indexando vóxeles coronarios...
✅ Indexación completa
Train batches: 72 | Val batches: 18


In [9]:
model     = ResUNetSegmentation().to(DEVICE)
model.load_pretrained_encoder(CHECKPOINT_ENCODER)

param_groups = model.get_param_groups(lr_encoder=LR_ENCODER, lr_decoder=LR_DECODER)
optimizer    = torch.optim.AdamW(param_groups, weight_decay=1e-5)
criterion    = DiceBCELoss(dice_weight=1.0, bce_weight=1.0)
scaler       = torch.amp.GradScaler('cuda')
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
                   optimizer, T_max=EPOCHS, eta_min=LR_MIN)

print(f"Modelo listo — {sum(p.numel() for p in model.parameters()):,} parámetros totales")

   Encoder cargado: 24 tensores
   Bloques: ['e1', 'e2', 'e3', 'e4']
   Decoder: inicialización aleatoria (kaiming)
Modelo listo — 5,686,465 parámetros totales


In [10]:
# Prueba de un solo batch antes de lanzar
model.train()
vol, mask = next(iter(train_loader))
vol, mask = vol.to(DEVICE), mask.to(DEVICE)

with torch.amp.autocast('cuda'):
    pred = model(vol)
    loss, ld, lb = criterion(pred, mask)

print(f"Batch OK — Loss: {loss.item():.4f} | Dice: {ld.item():.4f} | BCE: {lb.item():.4f}")
print(f"Pred shape: {pred.shape} | min: {pred.min():.3f} | max: {pred.max():.3f}")

Batch OK — Loss: 1.5718 | Dice: 0.9978 | BCE: 0.5741
Pred shape: torch.Size([2, 1, 128, 128, 128]) | min: -8.672 | max: 3.367


In [ ]:
writer       = SummaryWriter(TB_DIR)
best_dice    = 0.0
global_step  = 0

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    train_loss = train_dice = 0.0

    for vol, mask in tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS} train"):
        vol, mask = vol.to(DEVICE), mask.to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            pred = model(vol)
            loss, ld, lb = criterion(pred, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_dice += dice_score(pred, mask)
        writer.add_scalar("Loss/train_batch", loss.item(), global_step)
        global_step += 1

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)

    # ── VALIDACIÓN ─────────────────────────────────────────
    model.eval()
    val_loss = val_dice = 0.0

    with torch.no_grad():
        for vol, mask in tqdm(val_loader, desc=f"Ep {epoch}/{EPOCHS} val"):
            vol, mask = vol.to(DEVICE), mask.to(DEVICE)
            with torch.amp.autocast('cuda'):
                pred = model(vol)
                loss, _, _ = criterion(pred, mask)
            val_loss += loss.item()
            val_dice += dice_score(pred, mask)

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)

    scheduler.step()
    lr_enc = optimizer.param_groups[0]['lr']
    lr_dec = optimizer.param_groups[1]['lr']

    # ── TENSORBOARD ────────────────────────────────────────
    writer.add_scalars("Loss/epoch",      {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("Dice/epoch",      {"train": train_dice, "val": val_dice}, epoch)
    writer.add_scalars("LR",              {"encoder": lr_enc,   "decoder": lr_dec}, epoch)

    # ── CHECKPOINT ─────────────────────────────────────────
    torch.save(model.state_dict(),
               os.path.join(CKPT_DIR, f"model_epoch_{epoch}.pth"))

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(),
                   os.path.join(CKPT_DIR, "model_best.pth"))
        print(f"  ⭐ Nuevo mejor Dice: {best_dice:.4f}")

    print(f"Ep {epoch:03d} | "
          f"Loss train {train_loss:.4f} val {val_loss:.4f} | "
          f"Dice train {train_dice:.4f} val {val_dice:.4f} | "
          f"LR enc {lr_enc:.1e} dec {lr_dec:.1e}")

writer.close()
print(f"\n✅ Entrenamiento completado — Mejor Dice val: {best_dice:.4f}")